# Day 7 Advanced Lab: Traversal Architectures at 64-Iteration Scale (Single-Node Edition)

Welcome to the high-depth Day 7 performance showdown. Today, we scale our graph validation framework to exactly **64 consecutive iterations** to programmatically measure and contrast processing paradigms within a single local node context (`local[*]` mode).

### Traversal Profiles Under Review:
1. **Case 2: Native Recursive CTEs (64 Iterations):** Spark's native `UnionLoopExec` runs an internal execution loop. While it shields the driver from logical tree bloating, it executes standard unaligned partition distribution across your local CPU cores, generating continuous serialization cycles over 64 sequential steps.
2. **Case 3: Manual Linear Loop (64 Key-Based Repartitions):** We run a classic procedural loop through 64 steps but enforce physical thread alignment by explicitly calling key-targeted repartitioning right before the join to minimize arbitrary thread context switching.
3. **Case 4: Manual Logarithmic Loop ($2^i$ Pointer Jumping Technique):** We implement the exponential growth strategy. Instead of crawling step-by-step 64 times, every self-join doubles the distance traveled by our pointers ($1 \rightarrow 2 \rightarrow 4 \rightarrow 8 \rightarrow 16 \rightarrow 32 \rightarrow 64$). This strategy compresses the work from 64 agonizing linear shuffles down to **exactly 6 steps** ($2^6 = 64$).
4. **Case 5: Hybrid Paradigm (Manual Logarithmic Pre-Jump + Native Recursive CTE Cleanup):** We combine both worlds. We apply a short 3-iteration logarithmic jump to reduce chain lengths by 8x, then hand the compressed pointer layout over to a native recursive CTE to clean up the remaining long-tail links linearly without blowing up driver history buffers.

### Initialize the Single-Node Spark Session & Configure Version-Safe Telemetry Hooks

In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Allocate memory strictly to the single local Driver JVM process and fix partitions
spark = SparkSession.builder \
    .config('spark.driver.memory', '4g') \
    .config('spark.sql.shuffle.partitions', '250') \
    .getOrCreate()

def print_performance_invoice(case_name: str, duration: float):
    """Calculates and prints precision runtime metrics processed during the query."""
    print(f'\n[SINGLE-NODE METRICS INVOICE] --- {case_name} ---')
    print(f'Execution Runtime:          {duration:.4f} seconds')
    print('-----------------------------------------------------------------------')


### CASE 1: Building the Balanced High-Volume Source Grid
We construct a thick data framework where every layer of depth contains parallel records loaded with a large text payload column to make sure each data operation registers a clear storage cost.

In [2]:
print('--- Case 1: Fabricating wide relational source grid rows ---')
start_gen = time.time()

heavy_payload = 'LOCAL_THREAD_SHUFFLE_TAX_BYTE_CONGESTION_PAD_' * 500
records = [('EMP_0_0', None, 'Root Context.')]

for item in range(1000):
    records.append((f'EMP_1_{item}', 'EMP_0_0', heavy_payload))

for depth in range(2, 66): # Built up to a depth boundary of 65 continuous layers to sustain a 64-step run
    for item in range(1000):
        records.append((
            f'EMP_{depth}_{item}', 
            f'EMP_{depth-1}_{item}', 
            heavy_payload
        ))

source_tree_df = spark.createDataFrame(records, ['employee_id', 'manager_id', 'payload'])
source_tree_df.createOrReplaceTempView('corporate_grid')
source_tree_df.cache()
source_tree_df.count()

print(f'Data creation completed in {time.time() - start_gen:.4f} seconds | Row count: {source_tree_df.count()}')

--- Case 1: Fabricating wide relational source grid rows ---
Data creation completed in 30.8810 seconds | Row count: 65001


### CASE 2: Native Recursive CTEs (64 Iterations)
We execute a native SQL recursive query block targeted at 64 iterations deep. This evaluates the recursive dataset utilizing standard Catalyst execution graphs.

In [3]:
print('=== Case 2: Running Native SQL Recursive CTE (64 Iterations) ===')
start_time_c2 = time.time()

native_cte_df = spark.sql('''
WITH RECURSIVE native_recursive_loop AS (
  SELECT employee_id, manager_id, payload, 0 AS depth
  FROM corporate_grid WHERE manager_id IS NULL
  UNION ALL
  SELECT e.employee_id, e.manager_id, e.payload, nrl.depth + 1 AS depth
  FROM corporate_grid e
  JOIN native_recursive_loop nrl ON e.manager_id = nrl.employee_id
  WHERE nrl.depth < 64
)
SELECT depth, count(*) AS layer_count 
FROM native_recursive_loop 
GROUP BY depth 
ORDER BY depth DESC
''')

native_cte_df.show(5)
duration_c2 = time.time() - start_time_c2

print_performance_invoice('Case 2: Native SQL Recursive CTE', duration_c2)

=== Case 2: Running Native SQL Recursive CTE (64 Iterations) ===
+-----+-----------+
|depth|layer_count|
+-----+-----------+
|   64|       1000|
|   63|       1000|
|   62|       1000|
|   61|       1000|
|   60|       1000|
+-----+-----------+
only showing top 5 rows

[SINGLE-NODE METRICS INVOICE] --- Case 2: Native SQL Recursive CTE ---
Execution Runtime:          94.6204 seconds
-----------------------------------------------------------------------


### Spark UI Inspection Boundary: Post-Case 2
Open your local browser context to inspect the performance timeline of the native recursive stages.

In [4]:
print('⏲️  LAB PAUSE: Case 2 evaluation complete.')
print('Open your browser and inspect the internal JVM processing dashboard here:')
print('http://localhost:4040/')
print('\nLook for: The sheer quantity of distinct sequential stages generated inside the Jobs tab.')
print('Waiting 60 seconds before commencing Case 3 data execution...')
time.sleep(60)
print('Resuming laboratory execution matrix.')

⏲️  LAB PAUSE: Case 2 evaluation complete.
Open your browser and inspect the internal JVM processing dashboard here:
http://localhost:4040/

Look for: The sheer quantity of distinct sequential stages generated inside the Jobs tab.
Waiting 60 seconds before commencing Case 3 data execution...
Resuming laboratory execution matrix.


### CASE 3: Manual Linear Loop (Explicit Key-Based Repartitioning over 64 Steps)
Now we switch to a manual python loop structure pushed to exactly 64 steps. We pass the dataframe layout through a clean RDD reconstruction structure to safely drop Catalyst memory history chains.

In [5]:
print('=== Case 3: Running Manual Recursion with Key-Based Repartitioning (64 Steps) ===')
start_time_c3 = time.time()

frontier_df = spark.sql('SELECT employee_id, manager_id, payload FROM corporate_grid WHERE manager_id IS NULL')
base_lookup_df = spark.sql('SELECT employee_id, manager_id FROM corporate_grid')

print('Launching explicit key-aligned manual loop passes...')
for step in range(1, 65):
    aligned_base = base_lookup_df.repartition('manager_id').alias('b')
    aligned_frontier = frontier_df.repartition('employee_id').alias('f')
    
    frontier_df = aligned_base.join(
        aligned_frontier,
        F.col('b.manager_id') == F.col('f.employee_id'),
        'inner'
    ).select(
        F.col('b.employee_id').alias('employee_id'),
        F.col('b.manager_id').alias('manager_id'),
        F.col('f.payload').alias('payload')
    )
    
    # Truncate physical lineage graph using low-level RDD boundaries to stop Java StringBuilder crashes
    frontier_df = spark.createDataFrame(frontier_df.rdd, frontier_df.schema)
    frontier_df.persist()
    frontier_df.count()
    if step % 15 == 0 or step == 64:
        print(f'Completed Linear Traversal Step {step}/64 | Churning rows safely across local memory frames...')

duration_c3 = time.time() - start_time_c3

print_performance_invoice('Case 3: Key-Aligned Manual Linear Loop', duration_c3)

=== Case 3: Running Manual Recursion with Key-Based Repartitioning (64 Steps) ===
Launching explicit key-aligned manual loop passes...
Completed Linear Traversal Step 15/64 | Churning rows safely across local memory frames...
Completed Linear Traversal Step 30/64 | Churning rows safely across local memory frames...
Completed Linear Traversal Step 45/64 | Churning rows safely across local memory frames...
Completed Linear Traversal Step 60/64 | Churning rows safely across local memory frames...
Completed Linear Traversal Step 64/64 | Churning rows safely across local memory frames...

[SINGLE-NODE METRICS INVOICE] --- Case 3: Key-Aligned Manual Linear Loop ---
Execution Runtime:          257.2879 seconds
-----------------------------------------------------------------------


### Spark UI Inspection Boundary: Post-Case 3
Compare the local JVM memory overhead of explicit lineage truncation against the prior runs.

In [6]:
print('⏲️  LAB PAUSE: Case 3 (Key-Based Linear) evaluation complete.')
print('Open your browser and check the runtime properties inside your single-node session:')
print('http://localhost:4040/')
print('\nNote: Look at how much memory is actively occupied under the Storage tab.')
print('Waiting 60 seconds before commencing Case 4 data execution...')
time.sleep(60)
print('Resuming laboratory execution matrix.')

⏲️  LAB PAUSE: Case 3 (Key-Based Linear) evaluation complete.
Open your browser and check the runtime properties inside your single-node session:
http://localhost:4040/

Note: Look at how much memory is actively occupied under the Storage tab.
Waiting 60 seconds before commencing Case 4 data execution...
Resuming laboratory execution matrix.


### CASE 4: Manual Logarithmic Loop ($2^i$ Technique covering 64 depth range)
We now implement the exponential growth pattern established in our core optimizations. Instead of stepping forward linearly 64 times, we use a manual loop that joins the state reference back to itself. Each loop doubles our distance traveled ($2^1 \rightarrow 2^2 \rightarrow 2^3 \rightarrow 2^4 \rightarrow 2^5 \rightarrow 2^6$). This allows us to resolve the entire 64-layer deep lineage horizon in **exactly 6 loops**, cutting out massive serialization and data churning costs.

In [7]:
print('=== Case 4: Running Manual Logarithmic Pointer Jumping ===')
start_time_c4 = time.time()

spark.sql('''
  SELECT employee_id, manager_id as next_id 
  FROM corporate_grid 
  GROUP BY employee_id, manager_id
''').createOrReplaceTempView('step_0')

for i in range(1, 7):
    print(f'Computing Logarithmic Pointer Leap {i}/6 (Resolving up to distance {2**i})...')
    spark.sql(f'''
      SELECT 
        s1.employee_id, 
        coalesce(s2.next_id, s1.next_id) as next_id
      FROM step_{i-1} s1
      LEFT JOIN step_{i-1} s2 ON s1.next_id = s2.employee_id
    ''').createOrReplaceTempView(f'step_{i}')
    
    spark.table(f'step_{i}').persist().count()

duration_c4 = time.time() - start_time_c4

print_performance_invoice('Case 4: Logarithmic Pointer Jumping', duration_c4)
print(f'ACCELERATION STATEMENT: Logarithmic Jumping is {duration_c2 / max(0.001, duration_c4):.2f}x faster than Pure Native SQL CTEs.')
print(f'ACCELERATION STATEMENT: Logarithmic Jumping is {duration_c3 / max(0.001, duration_c4):.2f}x faster than Key-Aligned Linear Loops.')

=== Case 4: Running Manual Logarithmic Pointer Jumping ===
Computing Logarithmic Pointer Leap 1/6 (Resolving up to distance 2)...
Computing Logarithmic Pointer Leap 2/6 (Resolving up to distance 4)...
Computing Logarithmic Pointer Leap 3/6 (Resolving up to distance 8)...
Computing Logarithmic Pointer Leap 4/6 (Resolving up to distance 16)...
Computing Logarithmic Pointer Leap 5/6 (Resolving up to distance 32)...
Computing Logarithmic Pointer Leap 6/6 (Resolving up to distance 64)...

[SINGLE-NODE METRICS INVOICE] --- Case 4: Logarithmic Pointer Jumping ---
Execution Runtime:          24.0565 seconds
-----------------------------------------------------------------------
ACCELERATION STATEMENT: Logarithmic Jumping is 3.93x faster than Pure Native SQL CTEs.
ACCELERATION STATEMENT: Logarithmic Jumping is 10.70x faster than Key-Aligned Linear Loops.


### ⏲️ Spark UI Inspection Boundary: Post-Case 4
Observe the system state after the logarithmic tracking block finishes before initiating Case 5.

In [8]:
print('⏲️  LAB PAUSE: Case 4 (Logarithmic Jumps) evaluation complete.')
print('Open your browser to look at the storage state changes under your session threads:')
print('http://localhost:4040/')
print('\nObserve: Notice how execution complexity was cut down to single-digit stages.')
print('Waiting 60 seconds before starting Case 5 Hybrid paradigm evaluation...')
time.sleep(60)
print('Resuming laboratory execution matrix.')

⏲️  LAB PAUSE: Case 4 (Logarithmic Jumps) evaluation complete.
Open your browser to look at the storage state changes under your session threads:
http://localhost:4040/

Observe: Notice how execution complexity was cut down to single-digit stages.
Waiting 60 seconds before starting Case 5 Hybrid paradigm evaluation...
Resuming laboratory execution matrix.


### CASE 5: Hybrid Paradigm (Manual Logarithmic Pre-Jump + Native Recursive CTE Cleanup)
We now initialize a hybrid pipeline. We run a manual logarithmic loop for exactly 3 iterations to pre-jump chain pointers up to a distance of 8 ($2^3 = 8$). This compresses the deep lineages by 8x. Then, we feed this lightweight state reference map directly into a native SQL recursive CTE, allowing it to complete the remaining traversal steps linearly without hitting driver memory walls.

In [9]:
print('=== Case 5: Running Hybrid Paradigm (Logarithmic Pre-Jump + Native Recursive CTE) ===')
start_time_c5 = time.time()

# 1. Shallow manual logarithmic pre-jump loop (3 steps -> resolves up to distance 8)
spark.sql('''
  SELECT employee_id, manager_id as next_id 
  FROM corporate_grid 
  GROUP BY employee_id, manager_id
''').createOrReplaceTempView('hybrid_step_0')

for i in range(1, 4):
    print(f' [Hybrid Pre-Jump] Computing Logarithmic Step {i}/3 (Distance {2**i})...')
    spark.sql(f'''
      SELECT 
        s1.employee_id, 
        coalesce(s2.next_id, s1.next_id) as next_id
      FROM hybrid_step_{i-1} s1
      LEFT JOIN hybrid_step_{i-1} s2 ON s1.next_id = s2.employee_id
    ''').createOrReplaceTempView(f'hybrid_step_{i}')
    spark.table(f'hybrid_step_{i}').persist().count()

# 2. Native Recursive CTE clean up over the highly compressed graph reference map
print('Launching Native SQL Recursive CTE over the pre-jump map state...')
hybrid_cte_df = spark.sql('''
WITH RECURSIVE hybrid_recursive_cleanup AS (
  SELECT employee_id, next_id, 0 AS depth
  FROM hybrid_step_3
  UNION ALL
  SELECT h.employee_id, s.next_id, h.depth + 1 AS depth
  FROM hybrid_recursive_cleanup h
  JOIN hybrid_step_3 s ON h.next_id = s.employee_id
  WHERE h.next_id IS NOT NULL AND h.depth < 10
)
SELECT depth, count(*) AS layer_count 
FROM hybrid_recursive_cleanup 
GROUP BY depth 
ORDER BY depth DESC
''')

hybrid_cte_df.show(5)
duration_c5 = time.time() - start_time_c5

print_performance_invoice('Case 5: Hybrid Logarithmic + Recursive CTE', duration_c5)
print(f'ACCELERATION STATEMENT: Hybrid Processing is {duration_c2 / max(0.001, duration_c5):.2f}x faster than pure Native SQL CTEs.')

=== Case 5: Running Hybrid Paradigm (Logarithmic Pre-Jump + Native Recursive CTE) ===
 [Hybrid Pre-Jump] Computing Logarithmic Step 1/3 (Distance 2)...
 [Hybrid Pre-Jump] Computing Logarithmic Step 2/3 (Distance 4)...
 [Hybrid Pre-Jump] Computing Logarithmic Step 3/3 (Distance 8)...
Launching Native SQL Recursive CTE over the pre-jump map state...
+-----+-----------+
|depth|layer_count|
+-----+-----------+
|    9|       1000|
|    8|       9000|
|    7|      17000|
|    6|      25000|
|    5|      33000|
+-----+-----------+
only showing top 5 rows

[SINGLE-NODE METRICS INVOICE] --- Case 5: Hybrid Logarithmic + Recursive CTE ---
Execution Runtime:          67.6352 seconds
-----------------------------------------------------------------------
ACCELERATION STATEMENT: Hybrid Processing is 1.40x faster than pure Native SQL CTEs.


### Final Spark UI Verification Boundary
Analyze the structural performance layout after all processing tracks have settled.

In [10]:
print('⏲️  FINAL LAB PAUSE: Case 5 Hybrid validation lifecycle complete.')
print('Open your browser to check the ultimate state of your single-node session thread metrics:')
print('http://localhost:4040/')
print('\nObserve: Notice how combining exponential jumping with native recursion cuts down total timeline costs.')
print('Waiting 60 seconds for final session inspection before laboratory closes...')
time.sleep(60)
print('Lab completed successfully.')

⏲️  FINAL LAB PAUSE: Case 5 Hybrid validation lifecycle complete.
Open your browser to check the ultimate state of your single-node session thread metrics:
http://localhost:4040/

Observe: Notice how combining exponential jumping with native recursion cuts down total timeline costs.
Waiting 60 seconds for final session inspection before laboratory closes...
Lab completed successfully.


## 📊 Post-Lab Engineering Assessment: Single-Node Traversal Telemetry

By cross-referencing our wall-clock runtimes with the hard physical telemetry captured from the Spark UI Executors dashboard, we can unmask the literal JVM and engine-level behavior of each traversal architecture at a 64-iteration scale.

### 📈 Empirical Performance Scorecard

| Traversal Paradigm | Completed Tasks | Cumulative Task Time (GC Time) | Storage Memory Footprint | Persisted RDD Blocks | Shuffle Read / Shuffle Write |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Case 2: Native Recursive CTE** | 1,806 | 2.5 min (0.7 s) | 34.2 MiB | 12 | 817.9 MiB / 817.0 MiB |
| **Case 3: Key-Aligned Linear Loop** | 2,905 | 7.7 min (1.0 s) | 49.2 MiB | 76 | 896.8 MiB / 895.9 MiB |
| **Case 4: Logarithmic Pointer Jumping** | 7,187 | 9.0 min (2.0 s) | 69.9 MiB | 1,576 | 900.3 MiB / 899.4 MiB |
| **Case 5: Hybrid Pre-Jump + Native SQL** | 10,976 | 11.0 min (2.0 s) | 89.1 MiB | 1,576 | 933.2 MiB / 910.4 MiB |

---

### 🧠 Deep-Dive Architectural Breakdown

#### 1. The Minimalism of Native Recursion (Case 2)
* **The Telemetry:** 1,806 Tasks | 12 RDD Blocks | 34.2 MiB Storage Memory
* **The Reality:** Notice how the native engine maintains a remarkably lean memory profile, pinning only 12 RDD blocks despite completing 64 sequential iterations. This proves that `UnionLoopExec` aggressively truncates logical plan lineages and disposes of historical iteration data frames on every loop tick. However, it pays an unaligned partitioning price: it must execute 1,806 sequential, short-lived tasks that hammer local thread channels, dragging wall-clock performance out to nearly 95 seconds.

#### 2. The Heavy Cost of Linear Shuffle Churn (Case 3)
* **The Telemetry:** 2,905 Tasks | 7.7 min Cumulative Task Time | 896.8 MiB Shuffle Read
* **The Reality:** Reverting to a manual linear walk over 64 steps while programmatically forcing RDD reconstructions completely spikes our thread execution debt. The driver is forced to coordinate 2,905 total tasks, racking up an agonizing **7.7 minutes of cumulative CPU task time**. Because the machine must serialize, bucket-sort, write to disk, and deserialize raw binary payloads across local memory boundaries 64 independent times, local storage channels get completely choked, causing the worst wall-clock runtime in our benchmark (257.28 seconds).

#### 3. The Logarithmic Quantum Leap (Case 4)
* **The Telemetry:** 7,187 Tasks | 1,576 RDD Blocks | 69.9 MiB Storage Memory | 24.05s Wall-Clock
* **The Reality:** This case highlights a fascinating distributed processing paradox: it registers a massive jump to 7,187 completed tasks and hoards 1,576 RDD blocks in storage memory. This happens because joining an active pointer map against *itself* exponentially broadens the active physical graph layouts, forcing Spark to build wide, highly parallelized partition branches. Yet, because it compresses 64 linear passes into **exactly 6 elegant steps** ($2^6=64$), it completely minimizes sequential stage synchronization barriers. It maximizes multi-threaded CPU thread utilization, allowing it to finish **3.93× faster** than pure native SQL and **10.70× faster** than linear loops in wall-clock time.

#### 4. The Hybrid Safety Valve (Case 5)
* **The Telemetry:** 10,976 Tasks | 11.0 min Cumulative Task Time | 89.1 MiB Storage Memory
* **The Reality:** The hybrid configuration serves as the ultimate real-world production design for deep graph topologies. By utilizing a manual loop for the first 3 giant leaps, we legally bypass the database "No Selfies" rule to flatten and shrink pointer chains by 8x. While it hoards the highest memory footprint (89.1 MiB and 1,576 RDD blocks) to track both the manual cache states and the downstream native query states, it isolates the execution from driver-level logical plan crashes. It reduces the tail-end linear work significantly, successfully accelerating performance over the pure native baseline by **1.40×**.